# Task 14: Custom CUDA-accelerated Swish activation

Requires a CUDA-enabled GPU + CUDA Toolkit to build/run. Writes the kernel + extension files and shows how to compile and benchmark them.

In [1]:
%%writefile swish_kernel.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void swish_forward_kernel(const float* x, float* out, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        float val = x[idx];
        out[idx] = val / (1.0f + expf(-val));
    }
}

torch::Tensor swish_forward(torch::Tensor x) {
    auto out = torch::empty_like(x);
    int n = x.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    swish_forward_kernel<<<blocks, threads>>>(x.data_ptr<float>(), out.data_ptr<float>(), n);
    return out;
}


Writing swish_kernel.cu


In [2]:
%%writefile swish_binding.cpp
#include <torch/extension.h>

torch::Tensor swish_forward(torch::Tensor x);

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &swish_forward, "Swish forward (CUDA)");
}


Writing swish_binding.cpp


In [ ]:
import torch
from torch.utils.cpp_extension import load
import time

swish_cuda = load(
    name="swish_cuda",
    sources=["swish_binding.cpp", "swish_kernel.cu"],
    verbose=True
)

x = torch.randn(1_000_000, device="cuda")

torch.cuda.synchronize()
t0 = time.time()
out_custom = swish_cuda.forward(x)
torch.cuda.synchronize()
print("custom kernel:", time.time() - t0)

t0 = time.time()
out_ref = x * torch.sigmoid(x)
torch.cuda.synchronize()
print("torch builtin:", time.time() - t0)

print("max abs diff:", (out_custom - out_ref).abs().max().item())
